In [20]:
import pandas as pd
from pathlib import Path

csv_path_time = Path.home() / "Desktop" / "side_quest" / "PLC_permit_time.csv"
csv_path_plc = Path.home() / "Desktop" / "side_quest" / "PLC_permit.csv"
df = pd.read_csv(csv_path_time)
df2 = pd.read_csv(csv_path_plc)


In [ ]:

print("DF",df.columns.tolist())
print("DF2",df2.columns.tolist())



DF ['OBJECTID', 'Permit_Number', 'Issuance_Stage', 'Jurisdictional_Breakdown', 'Total_Occurrence', 'Total_Duration']
DF2 ['Permit_Number', 'Date_of_Submission', 'Date_of_Permit_Issuance', 'Completed_Date', 'Expiration_Date', 'Permit_Name', 'Work_Type', 'Primary_Work_Scope', 'Permit_Status', 'Civic_ID', 'Civic_Number', 'Street_Name', 'Community', 'PID', 'Estimated_Project_Value', 'Most_Recent_Inspection', 'Type_of_Structure', 'Occupancy_Type', 'Existing_Residential_Units', 'Total_End_Residential_Units', 'Building_Footprint_Area', 'Number_of_Storeys', 'Work_Description', 'Active_Parcel', 'Date_Inspection_Completed', 'District', 'Inspection_Outcome', 'Overall_Building_Height', 'Net_New_Units', 'OBJECTID']


In [18]:
by_stage = (
    df.groupby("Issuance_Stage")["Total_Duration"]
      .agg(["count", "median", "mean"])
      .sort_values("median", ascending=False)
)

print(by_stage)


                count   median        mean
Issuance_Stage                            
Other Timeline   5572  857.614  649.104386
Post Issuance   63205  156.850  294.757438
Pre Issuance    92499    7.364   48.724480


In [26]:
timeline=df
permits=df2
print("Timeline rows:", len(timeline))
print("Unique timeline permits:", timeline["Permit_Number"].nunique())

print("Permit rows:", len(permits))
print("Unique permit numbers:", permits["Permit_Number"].nunique())

Timeline rows: 161282
Unique timeline permits: 61430
Permit rows: 19588
Unique permit numbers: 17073


In [27]:
timeline_ids = set(timeline["Permit_Number"].dropna().astype(str).str.strip())
permit_ids = set(permits["Permit_Number"].dropna().astype(str).str.strip())

print("Shared permit IDs:", len(timeline_ids & permit_ids))
print("Only in timeline:", len(timeline_ids - permit_ids))
print("Only in permits:", len(permit_ids - timeline_ids))

Shared permit IDs: 16984
Only in timeline: 44446
Only in permits: 89


In [28]:
permits["Date_of_Submission"] = pd.to_datetime(
    permits["Date_of_Submission"], errors="coerce"
)
permits["Date_of_Permit_Issuance"] = pd.to_datetime(
    permits["Date_of_Permit_Issuance"], errors="coerce"
)

one_permit = permits.drop_duplicates("Permit_Number").copy()
one_permit["approval_days"] = (
    one_permit["Date_of_Permit_Issuance"]
    - one_permit["Date_of_Submission"]
).dt.days

issued = one_permit.dropna(subset=["approval_days"])

print(issued.groupby("Work_Type").agg(
    permits=("approval_days", "count"),
    median_days=("approval_days", "median"),
    p90_days=("approval_days", lambda x: x.quantile(0.9)),
))

              permits  median_days  p90_days
Work_Type                                   
Addition         1575         42.0     168.6
New Building     8730         21.0     103.1
Renovation       4571         25.0     100.0
